# Stage 2 XGBoost SHAP analysis (within-cohort CV)

This notebook computes out-of-fold SHAP values for the Stage 2 XGBoost model under the within-cohort setting, exports parquet artifacts, and generates SHAP figures for Liberman (Cohort A) and Buran (Cohort B).

Pipeline sections:
1. SHAP computation (10-fold subject-level CV)
2. Export (parquet)
3. Plotting (PNG + SVG)

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

from utils.benchmark_metrics import STAGE2_BEST_HP_DIR
from utils.liberman_classical import (
    liberman_feature_lists,
    stage2_feature_lists,
)
from utils.nn_stage2 import attach_noise_preds_long
from utils.nn_stage2_data import load_nn_stage2_data, splits_for_long_stage2
from utils.stage2_sklearn import _syn_prep
from utils.stage2_synthesis_cv import (
    build_cv_scenario_frames,
    fit_global_stage1_for_scenario,
    generate_cv_folds,
)

RANDOM_STATE = 22
N_SPLITS = 10
NOISE_LABEL_MODE = "predicted"
SCENARIO_BY_COHORT = {
    "liberman": "C",  # within-cohort for Liberman
    "buran": "A",  # within-cohort for Buran
}
COHORT_TO_EVAL = {
    "liberman": "Liberman",
    "buran": "Brad",
}

CACHE_DIR = Path("figures/cache/shap")
FIG_DIR = Path("figures/shap")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("startup cell loaded")

startup cell loaded


In [2]:
import seaborn as sns
import shap
import xgboost as xgb

from utils.benchmark_metrics import apply_slide_rcparams

apply_slide_rcparams()
sns.set_style("whitegrid")
print("shap version:", shap.__version__)

FEATURE_TYPE_ORDER = [
    "amplitude",
    "distance",
    "slope",
    "total_variance",
    "PeakIEarlyCurvature",
    "PeakICentralCurvature",
    "PeakILateCurvature",
    "TroughIEarlyCurvature",
    "TroughICentralCurvature",
    "TroughILateCurvature",
]


def _feature_annotations(raw_features: list[str]) -> pd.DataFrame:
    rows = []
    for f in raw_features:
        level_db = None
        feature_type = f
        is_noise_pred = f == "noise_preds"
        is_frequency = f == "frequency"
        if not (is_noise_pred or is_frequency):
            if "_" in f:
                base, suffix = f.rsplit("_", 1)
                try:
                    level_db = int(float(suffix))
                    feature_type = base
                except ValueError:
                    feature_type = f
            else:
                feature_type = f
        rows.append(
            {
                "feature_name": f,
                "feature_type": feature_type,
                "level_db": level_db,
                "is_stage1_noise_pred": bool(is_noise_pred),
                "is_frequency": bool(is_frequency),
            }
        )
    return pd.DataFrame(rows)


def _xgb_best_params_for_within(cohort_slug: str) -> dict[str, Any]:
    scen = SCENARIO_BY_COHORT[cohort_slug]
    hp_path = STAGE2_BEST_HP_DIR / f"{scen}.json"
    hp = json.loads(hp_path.read_text())
    return {
        k.replace("xgb__", ""): v
        for k, v in dict(hp["XGB"]["best_params"]).items()
    }


def _build_stage2_xgb_pipeline(
    syn_num: list[str],
    syn_cat: list[str],
    syn_log: list[str],
    best_params: dict[str, Any],
) -> Pipeline:
    pipe = Pipeline(
        [
            ("prep", _syn_prep(syn_num, syn_cat, syn_log)),
            ("xgb", xgb.XGBRegressor(random_state=1, n_jobs=-1)),
        ]
    )
    pipe.set_params(**{f"xgb__{k}": v for k, v in best_params.items()})
    return pipe


def _prep_feature_names(
    syn_num: list[str],
    syn_cat: list[str],
    syn_log: list[str],
    transformed_dim: int,
) -> list[str]:
    # _syn_prep preserves column counts for num/log branches and uses one-hot for cat.
    # In this notebook, cat is noise_preds (binary) with drop='first' => one output column.
    expected = list(syn_num) + list(syn_cat) + list(syn_log)
    if transformed_dim != len(expected):
        raise ValueError(
            "Transformed design width mismatch for Stage-2 features. "
            f"expected={len(expected)}, got={transformed_dim}"
        )
    return expected


def _collapse_by_feature_type(
    df_vals: pd.DataFrame,
    df_feat: pd.DataFrame,
    feature_table: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    by_type = feature_table.groupby("feature_type")["feature_name"].apply(list)
    val_out = {}
    feat_out = {}
    for ftype in FEATURE_TYPE_ORDER:
        cols = by_type.get(ftype, [])
        keep = [c for c in cols if c in df_vals.columns]
        if keep:
            val_out[ftype] = df_vals[keep].mean(axis=1)
            feat_out[ftype] = df_feat[keep].mean(axis=1)
    return pd.DataFrame(val_out, index=df_vals.index), pd.DataFrame(
        feat_out, index=df_feat.index
    )


FIG4_EXTRA_FEATURES = ("noise_preds", "frequency")
FIG4_ROW_ORDER = list(FEATURE_TYPE_ORDER) + list(FIG4_EXTRA_FEATURES)
FEATURE_PAPER_LABELS: dict[str, str] = {
    "amplitude": "Amplitude",
    "distance": "Peak-to-trough latency",
    "slope": "Wave I slope",
    "total_variance": "Total variance",
    "PeakIEarlyCurvature": "Peak I curvature (early)",
    "PeakICentralCurvature": "Peak I curvature (central)",
    "PeakILateCurvature": "Peak I curvature (late)",
    "TroughIEarlyCurvature": "Trough I curvature (early)",
    "TroughICentralCurvature": "Trough I curvature (central)",
    "TroughILateCurvature": "Trough I curvature (late)",
    "noise_preds": "Predicted noise",
    "frequency": "Frequency (kHz)",
}


def _build_fig4_featuretype_frames(
    df_vals: pd.DataFrame,
    df_feat: pd.DataFrame,
    feature_table: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Ten collapsed feature types plus noise_preds and frequency (12 rows)."""
    vals_type, feat_type = _collapse_by_feature_type(
        df_vals, df_feat, feature_table
    )
    for col in FIG4_EXTRA_FEATURES:
        if col in df_vals.columns:
            vals_type[col] = df_vals[col]
            feat_type[col] = df_feat[col]
    row_order = [c for c in FIG4_ROW_ORDER if c in vals_type.columns]
    rename = {c: FEATURE_PAPER_LABELS.get(c, c) for c in row_order}
    return (
        vals_type[row_order].rename(columns=rename),
        feat_type[row_order].rename(columns=rename),
    )


def _paper_label_from_raw_feature(raw_name: str) -> str:
    if raw_name in FEATURE_PAPER_LABELS:
        return FEATURE_PAPER_LABELS[raw_name]
    if "_" in raw_name:
        base, suffix = raw_name.rsplit("_", 1)
        try:
            level_db = int(float(suffix))
            base_label = FEATURE_PAPER_LABELS.get(base, base)
            return f"{base_label} ({level_db} dB)"
        except ValueError:
            return FEATURE_PAPER_LABELS.get(raw_name, raw_name)
    return FEATURE_PAPER_LABELS.get(raw_name, raw_name)


def _rename_columns_to_paper_labels(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(
        columns={c: _paper_label_from_raw_feature(c) for c in df.columns}
    )


def _build_level_group_frames(
    vals_raw: pd.DataFrame, feat_raw: pd.DataFrame, feature_table: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    level_rows = feature_table.dropna(subset=["level_db"]).copy()
    level_rows["level_db"] = level_rows["level_db"].astype(int)
    vals_out = {}
    feat_out = {}
    for lv in (50, 60, 70, 80):
        cols = level_rows.loc[
            level_rows["level_db"].eq(lv), "feature_name"
        ].tolist()
        vals_out[f"{lv}dB"] = vals_raw[cols].mean(axis=1)
        feat_out[f"{lv}dB"] = feat_raw[cols].mean(axis=1)
    return pd.DataFrame(vals_out, index=vals_raw.index), pd.DataFrame(
        feat_out, index=feat_raw.index
    )


def _save_current_fig(base_no_ext: Path, *, tight: bool = True) -> None:
    base_no_ext.parent.mkdir(parents=True, exist_ok=True)
    if tight:
        plt.tight_layout()
    plt.savefig(base_no_ext.with_suffix(".png"), dpi=300, bbox_inches="tight")
    plt.savefig(base_no_ext.with_suffix(".svg"), bbox_inches="tight")
    plt.close()


def _summary_beeswarm(
    values: pd.DataFrame,
    features: pd.DataFrame,
    title: str,
    out_base: Path,
    max_display: int = 10,
) -> None:
    shap.summary_plot(
        values.values,
        features=features.values,
        feature_names=list(values.columns),
        max_display=max_display,
        show=False,
        plot_type="dot",
    )
    plt.title(title)
    _save_current_fig(out_base)


def _summary_bar(
    values: pd.DataFrame,
    features: pd.DataFrame,
    title: str,
    out_base: Path,
    max_display: int = 10,
) -> None:
    shap.summary_plot(
        values.values,
        features=features.values,
        feature_names=list(values.columns),
        max_display=max_display,
        show=False,
        plot_type="bar",
    )
    plt.title(title)
    _save_current_fig(out_base)


def _interaction_pair_batched(
    explainer: shap.TreeExplainer,
    x_eval_t,
    i_left: int,
    i_right: int,
    *,
    batch_size: int = 64,
) -> np.ndarray:
    """Compute only one interaction pair in small batches to avoid OOM crashes."""
    n = x_eval_t.shape[0]
    out = np.empty(n, dtype=float)
    for s in range(0, n, batch_size):
        e = min(s + batch_size, n)
        inter = explainer.shap_interaction_values(x_eval_t[s:e])
        out[s:e] = inter[:, i_left, i_right]
    return out


def _xgb_booster_shap_values(
    booster,
    x_eval_t,
    *,
    i_noise: int,
    i_freq: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Fallback path using XGBoost native contribution/interactions predictions."""
    dm = xgb.DMatrix(x_eval_t)
    contrib = booster.predict(dm, pred_contribs=True)
    # drop bias term (last column)
    shap_vals = contrib[:, :-1]

    inter = booster.predict(dm, pred_interactions=True)
    pair = inter[:, i_noise, i_freq]
    return shap_vals, pair

shap version: 0.49.1


In [3]:
# Part 1: SHAP computation

data = load_nn_stage2_data()
splits = splits_for_long_stage2(data)

feats = liberman_feature_lists(data.reformatted_orig, data.common_cols)
syn_num, syn_log, syn_cat, _, _ = stage2_feature_lists(
    feats, noise_label=NOISE_LABEL_MODE
)

raw_feature_names = list(syn_num) + list(syn_cat) + list(syn_log)
feature_names_df = _feature_annotations(raw_feature_names)
n_wide_features = len(raw_feature_names)
print(f"Wide stage-2 features: {n_wide_features}")
assert n_wide_features >= 42, f"Too few wide features: {n_wide_features}"

# Fit global Stage 1 labels per scenario once, then reuse fold-wise.
global_s1 = {
    scen: fit_global_stage1_for_scenario(scen, data, splits, verbose=False)
    for scen in ("A", "B", "C")
}

cohort_outputs: dict[str, dict[str, pd.DataFrame]] = {}
for cohort_slug, eval_cohort in COHORT_TO_EVAL.items():
    scen = SCENARIO_BY_COHORT[cohort_slug]
    best_params = _xgb_best_params_for_within(cohort_slug)

    wide_all = (
        data.reformatted.reset_index(drop=True)
        if eval_cohort == "Brad"
        else data.reformatted_orig.reset_index(drop=True)
    )

    folds = generate_cv_folds(
        wide_all, n_splits=N_SPLITS, random_state=RANDOM_STATE
    )

    shap_rows = []
    feat_rows = []
    meta_rows = []
    interaction_rows = []

    for fold_id, (_tr_idx, _te_idx, tr_anim, te_anim) in enumerate(folds):
        w_tr, w_ev, _l_tr, _l_ev, _, _ = build_cv_scenario_frames(
            eval_cohort, scen, tr_anim, te_anim, data, splits
        )
        gs1 = global_s1[scen]
        w_aug_tr = attach_noise_preds_long(
            w_tr,
            gs1["animal_preds"],
            fallback_noise_cat=False,
            require_full_coverage=True,
        )
        w_aug_ev = attach_noise_preds_long(
            w_ev,
            gs1["animal_preds"],
            fallback_noise_cat=False,
        )

        train_df = w_aug_tr[w_aug_tr["DataGroup"].eq("Train")].reset_index(
            drop=True
        )
        x_train_raw = train_df[raw_feature_names].dropna()
        y_train = train_df.loc[x_train_raw.index, "synapses"].astype(float)

        x_eval_raw = w_aug_ev[raw_feature_names].dropna()
        eval_meta = w_aug_ev.loc[
            x_eval_raw.index,
            [
                "animal_id",
                "frequency",
                "noise_cat",
                "noise_preds",
                "synapses",
                "experimental_group",
            ],
        ].copy()

        model = _build_stage2_xgb_pipeline(
            syn_num, syn_cat, syn_log, best_params
        )
        model.fit(x_train_raw, y_train)

        prep = model.named_steps["prep"]
        x_eval_t = prep.transform(x_eval_raw)
        feat_names = _prep_feature_names(
            syn_num,
            syn_cat,
            syn_log,
            transformed_dim=int(x_eval_t.shape[1]),
        )

        i_noise = feat_names.index("noise_preds")
        i_freq = feat_names.index("frequency")

        try:
            explainer = shap.TreeExplainer(model.named_steps["xgb"])
            shap_vals = explainer.shap_values(x_eval_t)
            pair = _interaction_pair_batched(
                explainer,
                x_eval_t,
                i_noise,
                i_freq,
                batch_size=32,
            )
        except Exception as exc:
            print(
                f"TreeExplainer fallback ({cohort_slug}, fold {fold_id}): {type(exc).__name__}: {exc}"
            )
            shap_vals, pair = _xgb_booster_shap_values(
                model.named_steps["xgb"].get_booster(),
                x_eval_t,
                i_noise=i_noise,
                i_freq=i_freq,
            )

        shap_df = pd.DataFrame(
            shap_vals, columns=feat_names, index=x_eval_raw.index
        )
        feat_df = pd.DataFrame(
            x_eval_raw.values, columns=feat_names, index=x_eval_raw.index
        )
        interaction_df = pd.DataFrame(
            {
                "noisepred_x_frequency_interaction": pair,
            },
            index=x_eval_raw.index,
        )

        eval_meta["fold_id"] = fold_id
        eval_meta["cohort"] = cohort_slug

        shap_rows.append(shap_df)
        feat_rows.append(feat_df)
        meta_rows.append(eval_meta)
        interaction_rows.append(interaction_df)

    shap_oof = pd.concat(shap_rows).sort_index()
    feat_oof = pd.concat(feat_rows).sort_index()
    meta_oof = pd.concat(meta_rows).sort_index()
    inter_oof = pd.concat(interaction_rows).sort_index()

    common_idx = shap_oof.index.intersection(meta_oof.index).intersection(
        inter_oof.index
    )
    shap_oof = shap_oof.loc[common_idx].reset_index(drop=True)
    feat_oof = feat_oof.loc[common_idx].reset_index(drop=True)
    meta_oof = meta_oof.loc[common_idx].reset_index(drop=True)
    inter_oof = inter_oof.loc[common_idx].reset_index(drop=True)

    cohort_outputs[cohort_slug] = {
        "shap": shap_oof,
        "feat": feat_oof,
        "meta": meta_oof,
        "interaction": inter_oof,
        "interaction_pair": ("noise_preds", "frequency"),
    }

    print(
        cohort_slug,
        "rows=",
        len(meta_oof),
        "animals=",
        meta_oof["animal_id"].nunique(),
        "freqs=",
        sorted(meta_oof["frequency"].unique().tolist()),
    )

Wide stage-2 features: 43
TreeExplainer fallback (liberman, fold 0): ValueError: could not convert string to float: '[1.4843141E1]'
TreeExplainer fallback (liberman, fold 1): ValueError: could not convert string to float: '[1.4776135E1]'
TreeExplainer fallback (liberman, fold 2): ValueError: could not convert string to float: '[1.4811223E1]'
TreeExplainer fallback (liberman, fold 3): ValueError: could not convert string to float: '[1.4930488E1]'
TreeExplainer fallback (liberman, fold 4): ValueError: could not convert string to float: '[1.47950735E1]'
TreeExplainer fallback (liberman, fold 5): ValueError: could not convert string to float: '[1.4934652E1]'
TreeExplainer fallback (liberman, fold 6): ValueError: could not convert string to float: '[1.4894765E1]'
TreeExplainer fallback (liberman, fold 7): ValueError: could not convert string to float: '[1.4877647E1]'
TreeExplainer fallback (liberman, fold 8): ValueError: could not convert string to float: '[1.4995899E1]'
TreeExplainer fallb

In [4]:
# Part 2: export parquet

feature_names_df.to_parquet(CACHE_DIR / "feature_names.parquet", index=False)

for cohort_slug, payload in cohort_outputs.items():
    shap_df = payload["shap"].copy()
    meta_df = payload["meta"].copy()

    # Avoid duplicate parquet column names: row key `frequency` and SHAP feature `frequency`.
    shap_export_features = shap_df.rename(
        columns={"frequency": "frequency_feature"}
    )

    # One row per animal x frequency with SHAP columns and join keys.
    shap_export = pd.concat(
        [meta_df[["animal_id", "frequency"]], shap_export_features], axis=1
    )
    if not shap_export.columns.is_unique:
        dupes = shap_export.columns[shap_export.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate columns in shap export: {dupes}")

    shap_export.to_parquet(
        CACHE_DIR / f"shap_values_{cohort_slug}.parquet", index=False
    )

    meta_export = meta_df[
        [
            "animal_id",
            "frequency",
            "noise_cat",
            "noise_preds",
            "synapses",
            "fold_id",
            "cohort",
            "experimental_group",
        ]
    ].copy()
    meta_export.to_parquet(
        CACHE_DIR / f"shap_metadata_{cohort_slug}.parquet", index=False
    )

    print("exported", cohort_slug, len(meta_export), "rows")

exported liberman 616 rows
exported buran 360 rows


In [5]:
from utils.stage2_xgb_shap import (
    aggregated_feature_type_comparison_table,
    full_feature_comparison_table,
)

FULL_DISPLAY_COLS = {
    "feature_label": "Feature name",
    "cohort_a_mean_abs_shap": "Cohort A mean |SHAP|",
    "cohort_a_rank": "Cohort A rank",
    "cohort_b_mean_abs_shap": "Cohort B mean |SHAP|",
    "cohort_b_rank": "Cohort B rank",
}
AGG_DISPLAY_COLS = {
    "feature_type_label": "Feature type",
    "cohort_a_mean_abs_shap": "Cohort A mean |SHAP|",
    "cohort_a_rank": "Cohort A rank",
    "cohort_b_mean_abs_shap": "Cohort B mean |SHAP|",
    "cohort_b_rank": "Cohort B rank",
}

full_tbl = full_feature_comparison_table(cohort_outputs, feature_names_df)
agg_tbl = aggregated_feature_type_comparison_table(
    cohort_outputs, feature_names_df
)

assert len(full_tbl) >= 42
assert full_tbl["cohort_a_rank"].between(1, len(full_tbl)).all()
assert full_tbl["cohort_b_rank"].between(1, len(full_tbl)).all()


def _format_display(tbl, col_map):
    out = tbl[list(col_map)].rename(columns=col_map).copy()
    for c in ("Cohort A mean |SHAP|", "Cohort B mean |SHAP|"):
        out[c] = out[c].map("{:.5f}".format)
    return out


print(f"Full features (n={len(full_tbl)}, sorted by pooled mean |SHAP|)")
display(_format_display(full_tbl, FULL_DISPLAY_COLS))

print(f"Aggregated feature types (n={len(agg_tbl)})")
display(_format_display(agg_tbl, AGG_DISPLAY_COLS))

Full features (n=43, sorted by pooled mean |SHAP|)


,Feature name,Cohort A mean |SHAP|,Cohort A rank,Cohort B mean |SHAP|,Cohort B rank
0,Predicted noise,1.03179,1,0.80711,2
1,Frequency (kHz),0.65741,2,0.78100,3
2,Amplitude (60 dB),0.04460,23,0.95143,1
3,Total variance (60 dB),0.10727,9,0.30449,4
4,Strain (CBA/CaJ vs C57BL/6J),0.22860,3,0.00000,43
5,Amplitude (50 dB),0.19642,4,0.02030,34
6,Trough I curvature (early) (70 dB),0.07979,11,0.20682,7
7,Total variance (80 dB),0.01049,43,0.29521,5
8,Peak I curvature (late) (70 dB),0.15456,6,0.04122,24
9,Peak I curvature (early) (50 dB),0.16548,5,0.01477,38


Aggregated feature types (n=13)


,Feature type,Cohort A mean |SHAP|,Cohort A rank,Cohort B mean |SHAP|,Cohort B rank
0,Predicted noise,1.03179,1,0.80711,2
1,Frequency (kHz),0.65741,2,0.78100,3
2,Amplitude,0.28544,5,1.36785,1
3,Total variance,0.25621,7,0.73049,4
4,Trough I curvature (early),0.28261,6,0.37243,5
5,Peak I curvature (early),0.31662,3,0.23419,7
6,Peak I curvature (late),0.29924,4,0.20376,9
7,Wave I slope,0.21991,9,0.20603,8
8,Trough I curvature (late),0.17951,12,0.24250,6
9,Trough I curvature (central),0.18661,11,0.15902,10


In [6]:
# Part 3: plotting

NOISE_EXPOSED_COLOR = "#E69F00"
UNEXPOSED_COLOR = "#009E73"


def _top_cols_by_abs_mean(df_vals: pd.DataFrame, n: int) -> list[str]:
    s = df_vals.abs().mean(axis=0).sort_values(ascending=False)
    return s.head(n).index.tolist()


def _true_noise_point_colors(meta: pd.DataFrame) -> np.ndarray:
    exposed = meta["noise_cat"].astype(int).eq(1).values
    colors = np.empty(len(meta), dtype=object)
    colors[exposed] = NOISE_EXPOSED_COLOR
    colors[~exposed] = UNEXPOSED_COLOR
    return colors


def _noise_group_legend_handles() -> list:
    from matplotlib.lines import Line2D

    return [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=NOISE_EXPOSED_COLOR,
            markeredgecolor=NOISE_EXPOSED_COLOR,
            markersize=8,
            label="Noise-exposed",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=UNEXPOSED_COLOR,
            markeredgecolor=UNEXPOSED_COLOR,
            markersize=8,
            label="Unexposed",
        ),
    ]


def _beeswarm_dot_y_offsets(
    shaps: np.ndarray, *, row_height: float = 0.4
) -> np.ndarray:
    nbins = 100
    quant = np.round(
        nbins * (shaps - np.min(shaps)) / (np.max(shaps) - np.min(shaps) + 1e-8)
    )
    inds = np.argsort(quant + np.random.randn(len(shaps)) * 1e-6)
    layer = 0
    last_bin = -1
    ys = np.zeros(len(shaps))
    for ind in inds:
        if quant[ind] != last_bin:
            layer = 0
        ys[ind] = np.ceil(layer / 2) * ((layer % 2) * 2 - 1)
        layer += 1
        last_bin = quant[ind]
    ys *= 0.9 * (row_height / np.max(ys + 1))
    return ys


def _summary_beeswarm_true_noise_color(
    values: pd.DataFrame,
    features: pd.DataFrame,
    meta: pd.DataFrame,
    out_base: Path,
    *,
    max_display: int = 10,
) -> None:
    """Beeswarm matching feature-type summary, dots colored by true noise group."""
    cols = [c for c in FEATURE_TYPE_ORDER if c in values.columns]
    shap_vals = values[cols].values
    feature_names = [FEATURE_PAPER_LABELS.get(c, c) for c in cols]
    dot_colors = _true_noise_point_colors(meta)

    feature_order = np.argsort(np.sum(np.abs(shap_vals), axis=0))
    feature_order = feature_order[-min(max_display, len(feature_order)) :]

    row_height = 0.4
    plt.figure(
        figsize=(8, min(len(feature_order), max_display) * row_height + 1.5)
    )
    ax = plt.gca()
    ax.axvline(x=0, color="#999999", zorder=-1)

    for pos, i in enumerate(feature_order):
        ax.axhline(y=pos, color="#cccccc", lw=0.5, dashes=(1, 5), zorder=-1)
        shaps = shap_vals[:, i]
        inds = np.arange(len(shaps))
        np.random.shuffle(inds)
        shaps = shaps[inds]
        colors = dot_colors[inds]
        ys = _beeswarm_dot_y_offsets(shaps, row_height=row_height)
        ax.scatter(
            shaps,
            pos + ys,
            c=colors,
            s=16,
            alpha=1.0,
            linewidth=0,
            zorder=3,
            rasterized=len(shaps) > 500,
        )

    ax.set_yticks(
        range(len(feature_order)), [feature_names[i] for i in feature_order]
    )
    ax.tick_params(axis="y", labelsize=13, length=20, width=0.5)
    ax.tick_params(axis="x", labelsize=11)
    ax.set_ylim(-1, len(feature_order))
    ax.set_xlabel("SHAP value")
    for spine in ("right", "top", "left"):
        ax.spines[spine].set_visible(False)
    ax.yaxis.set_ticks_position("none")
    ax.xaxis.set_ticks_position("bottom")
    ax.legend(
        handles=_noise_group_legend_handles(),
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False,
        fontsize=11,
    )
    _save_current_fig(out_base, tight=True)


def _plot_level_grouped_bar(vals_level: pd.DataFrame, cohort_slug: str) -> None:
    cols = ["50dB", "60dB", "70dB", "80dB"]
    means = vals_level[cols].abs().mean(axis=0)
    plt.figure(figsize=(8, 5))
    plt.bar(cols, means.values, color="#4c72b0")
    plt.ylabel("Mean |SHAP|")
    plt.title(f"Mean absolute SHAP grouped by level ({cohort_slug})")
    _save_current_fig(FIG_DIR / f"shap_bar_bylevel_{cohort_slug}")


DEPENDENCE_GROUPS: list[list[str]] = [
    ["amplitude", "total_variance", "slope", "distance"],
    [
        "PeakIEarlyCurvature",
        "PeakICentralCurvature",
        "PeakILateCurvature",
    ],
    [
        "TroughIEarlyCurvature",
        "TroughICentralCurvature",
        "TroughILateCurvature",
    ],
]


def _plot_dependence_grouped_panel(
    vals_type: pd.DataFrame,
    feat_type: pd.DataFrame,
    meta: pd.DataFrame,
    cohort_slug: str,
    *,
    color_by: str,
    out_base: Path,
) -> Path:
    """Ten-panel dependence figure in three visually separated groups."""
    from matplotlib.gridspec import GridSpec

    use_noise_legend = color_by == "noise_preds"
    if color_by == "noise_preds":
        point_colors = _true_noise_point_colors(meta)
    elif color_by == "frequency":
        color_vals = meta["frequency"].astype(float).values
        cmap = "viridis"
        cbar_label = "Frequency (kHz)"
        vmin, vmax = float(np.nanmin(color_vals)), float(np.nanmax(color_vals))
    else:
        raise ValueError(f"Unsupported color_by={color_by!r}")

    # Three rows: 4 scalar | 3 peak | 3 trough
    fig = plt.figure(figsize=(10.5, 8.5))
    spec = GridSpec(
        3,
        4,
        figure=fig,
        height_ratios=[1.0, 1.0, 1.0],
        hspace=0.55,
        wspace=0.38,
    )

    panel_slots: list[tuple[int, int, str]] = []
    for col, ftype in enumerate(DEPENDENCE_GROUPS[0]):
        panel_slots.append((0, col, ftype))
    for col, ftype in enumerate(DEPENDENCE_GROUPS[1]):
        panel_slots.append((1, col, ftype))
    for col, ftype in enumerate(DEPENDENCE_GROUPS[2]):
        panel_slots.append((2, col, ftype))

    last_sc = None
    axes_by_row: dict[int, list] = {0: [], 1: [], 2: []}
    for row, col, ftype in panel_slots:
        ax = fig.add_subplot(spec[row, col])
        axes_by_row[row].append(ax)
        if ftype not in vals_type.columns:
            ax.set_visible(False)
            continue
        if use_noise_legend:
            last_sc = ax.scatter(
                feat_type[ftype].values,
                vals_type[ftype].values,
                c=point_colors,
                s=14,
                alpha=0.72,
                linewidths=0,
            )
        else:
            last_sc = ax.scatter(
                feat_type[ftype].values,
                vals_type[ftype].values,
                c=color_vals,
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                s=14,
                alpha=0.72,
                linewidths=0,
            )
        ax.set_xlabel(FEATURE_PAPER_LABELS.get(ftype, ftype))
        if col > 0:
            ax.set_yticklabels([])
        else:
            ax.set_ylabel("SHAP")

    if last_sc is None:
        raise ValueError(f"No dependence panels rendered for {cohort_slug}")

    # Subtle horizontal dividers between the three feature groups
    fig.canvas.draw()
    for row in (0, 1):
        top_axes = axes_by_row[row]
        bot_axes = axes_by_row[row + 1]
        if not top_axes or not bot_axes:
            continue
        y_line = 0.5 * (
            min(ax.get_position().y0 for ax in top_axes)
            + max(ax.get_position().y1 for ax in bot_axes)
        )
        fig.add_artist(
            plt.Line2D(
                [0.06, 0.86],
                [y_line, y_line],
                transform=fig.transFigure,
                color="#888888",
                linewidth=0.8,
                alpha=0.65,
                zorder=0,
            )
        )

    if use_noise_legend:
        fig.legend(
            handles=_noise_group_legend_handles(),
            loc="center left",
            bbox_to_anchor=(0.91, 0.5),
            frameon=False,
            fontsize=11,
        )
        fig.subplots_adjust(left=0.08, right=0.87, bottom=0.07, top=0.98)
    else:
        cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.72])
        cbar = fig.colorbar(last_sc, cax=cbar_ax)
        cbar.set_label(cbar_label)
        fig.subplots_adjust(left=0.08, right=0.87, bottom=0.07, top=0.98)
    _save_current_fig(out_base, tight=False)
    return out_base


def _plot_dependence_sets(
    vals_type: pd.DataFrame,
    feat_type: pd.DataFrame,
    meta: pd.DataFrame,
    cohort_slug: str,
) -> list[Path]:
    outputs = [
        _plot_dependence_grouped_panel(
            vals_type,
            feat_type,
            meta,
            cohort_slug,
            color_by="noise_preds",
            out_base=FIG_DIR / f"shap_dependence_noisepred_{cohort_slug}",
        ),
        _plot_dependence_grouped_panel(
            vals_type,
            feat_type,
            meta,
            cohort_slug,
            color_by="frequency",
            out_base=FIG_DIR / f"shap_dependence_frequency_{cohort_slug}",
        ),
    ]
    return outputs


def _plot_noisepred_frequency_interaction(
    interaction_df: pd.DataFrame,
    meta: pd.DataFrame,
    cohort_slug: str,
) -> Path:
    out = FIG_DIR / f"shap_interaction_noisepred_frequency_{cohort_slug}"
    plt.figure(figsize=(8, 6))
    sc = plt.scatter(
        meta["frequency"].astype(float).values,
        interaction_df["noisepred_x_frequency_interaction"].values,
        c=meta["noise_preds"].astype(float).values,
        cmap="coolwarm",
        alpha=0.75,
        s=20,
    )
    plt.xlabel("frequency")
    plt.ylabel("SHAP interaction(noise_preds × frequency)")
    plt.title(f"noise_preds × frequency interaction ({cohort_slug})")
    cb = plt.colorbar(sc)
    cb.ax.set_ylabel("noise_preds")
    _save_current_fig(out)
    return out

In [7]:
plot_artifacts: dict[str, list[str]] = {}

for cohort_slug, payload in cohort_outputs.items():
    vals_raw = payload["shap"]
    feat_raw = payload["feat"]
    meta = payload["meta"]
    inter_df = payload["interaction"]

    vals_type, feat_type = _collapse_by_feature_type(
        vals_raw, feat_raw, feature_names_df
    )
    vals_level, feat_level = _build_level_group_frames(
        vals_raw, feat_raw, feature_names_df
    )

    created = []

    # Figure 4: 12-row beeswarm (10 feature types + noise_preds + frequency)
    vals_fig4, feat_fig4 = _build_fig4_featuretype_frames(
        vals_raw, feat_raw, feature_names_df
    )
    p = FIG_DIR / f"shap_beeswarm_featuretypes_{cohort_slug}"
    _summary_beeswarm(
        vals_fig4,
        feat_fig4,
        f"Feature types + noise_preds + frequency ({cohort_slug})",
        p,
        max_display=len(vals_fig4.columns),
    )
    created.append(str(p))

    # 2. Bar — all 10 feature types aggregated across levels
    p = FIG_DIR / f"shap_bar_featuretypes_{cohort_slug}"
    _summary_bar(
        vals_type,
        feat_type,
        f"All feature types mean |SHAP| ({cohort_slug})",
        p,
        max_display=10,
    )
    created.append(str(p))

    # 3/4. Top 20 out of all wide features (paper labels)
    top20 = _top_cols_by_abs_mean(vals_raw, 20)
    vals_top20 = _rename_columns_to_paper_labels(vals_raw[top20])
    feat_top20 = _rename_columns_to_paper_labels(feat_raw[top20])

    p = FIG_DIR / f"shap_beeswarm_top20all42_{cohort_slug}"
    _summary_beeswarm(
        vals_top20,
        feat_top20,
        f"Top 20 of all {n_wide_features} features ({cohort_slug})",
        p,
        max_display=20,
    )
    created.append(str(p))

    p = FIG_DIR / f"shap_bar_top20all42_{cohort_slug}"
    _summary_bar(
        vals_top20,
        feat_top20,
        f"Top 20 of all {n_wide_features} mean |SHAP| ({cohort_slug})",
        p,
        max_display=20,
    )
    created.append(str(p))

    # 5/6. Grouped by level (4 groups)
    p = FIG_DIR / f"shap_beeswarm_bylevel_{cohort_slug}"
    _summary_beeswarm(
        vals_level,
        feat_level,
        f"Grouped by level ({cohort_slug})",
        p,
        max_display=4,
    )
    created.append(str(p))

    _plot_level_grouped_bar(vals_level, cohort_slug)
    created.append(str(FIG_DIR / f"shap_bar_bylevel_{cohort_slug}"))

    # 5. Feature-type beeswarm, both noise groups overlaid (true noise color)
    p = FIG_DIR / f"shap_beeswarm_featuretypes_noisecolor_{cohort_slug}"
    _summary_beeswarm_true_noise_color(
        vals_type,
        feat_type,
        meta,
        p,
        max_display=10,
    )
    created.append(str(p))

    # 10/11. Grouped dependence panels (10 feature types × 2 color modes)
    dep_outs = _plot_dependence_sets(vals_type, feat_type, meta, cohort_slug)
    created.extend([str(x) for x in dep_outs])

    # 12. SHAP interaction plot noise_preds × frequency
    inter_out = _plot_noisepred_frequency_interaction(
        inter_df, meta, cohort_slug
    )
    created.append(str(inter_out))

    # 13-16. Supplementary beeswarms per level (10 feature types)
    for lv in (50, 60, 70, 80):
        rows = feature_names_df[feature_names_df["level_db"].eq(lv)]
        level_cols = rows["feature_name"].tolist()
        rename_map = dict(zip(level_cols, rows["feature_type"]))
        vals_l = vals_raw[level_cols].rename(columns=rename_map)
        feat_l = feat_raw[level_cols].rename(columns=rename_map)
        p = FIG_DIR / f"shap_beeswarm_level{lv}_{cohort_slug}"
        _summary_beeswarm(
            vals_l,
            feat_l,
            f"{lv} dB feature-type beeswarm ({cohort_slug})",
            p,
            max_display=10,
        )
        created.append(str(p))

    plot_artifacts[cohort_slug] = created
    print("plots done for", cohort_slug, "count=", len(created))

plots done for liberman count= 14
plots done for buran count= 14


In [8]:
# Validation + digest

EXPECTED_PLOTS_PER_COHORT = 14  # 14 outputs x (png, svg)

all_files = []
for cohort_slug, payload in cohort_outputs.items():
    shap_df = payload["shap"]
    meta_df = payload["meta"]
    inter_df = payload["interaction"]

    assert (
        shap_df.shape[1] == n_wide_features
    ), f"{cohort_slug}: expected {n_wide_features} SHAP columns, got {shap_df.shape[1]}"
    assert (
        len(shap_df) == len(meta_df) == len(inter_df)
    ), f"{cohort_slug}: row mismatch among shap/meta/interaction"

    key_counts = meta_df.assign(
        _k=meta_df["animal_id"].astype(str)
        + "|"
        + meta_df["frequency"].astype(str)
    )["_k"].value_counts()
    assert (
        key_counts.max() == 1
    ), f"{cohort_slug}: duplicate animal_id x frequency rows in OOF output"

    # Plot artifact existence
    bases = plot_artifacts[cohort_slug]
    expected_count = len(bases)
    assert expected_count == EXPECTED_PLOTS_PER_COHORT, (
        f"{cohort_slug}: expected {EXPECTED_PLOTS_PER_COHORT} plot bases, "
        f"got {expected_count}"
    )
    cohort_files = []
    for b in bases:
        png = Path(f"{b}.png")
        svg = Path(f"{b}.svg")
        assert png.is_file(), f"Missing {png}"
        assert svg.is_file(), f"Missing {svg}"
        cohort_files.extend([png, svg])
    assert (
        len(cohort_files) == EXPECTED_PLOTS_PER_COHORT * 2
    ), f"{cohort_slug}: unexpected plot file count"
    all_files.extend(cohort_files)

    fold_cov = meta_df.groupby("fold_id")["animal_id"].nunique().to_dict()
    print(f"\n[{cohort_slug}]")
    print(" rows:", len(meta_df))
    print(" animals:", meta_df["animal_id"].nunique())
    print(" fold animal counts:", fold_cov)
    print(" interaction pair:", payload["interaction_pair"])
    print(" shap parquet:", CACHE_DIR / f"shap_values_{cohort_slug}.parquet")
    print(
        " metadata parquet:", CACHE_DIR / f"shap_metadata_{cohort_slug}.parquet"
    )

assert len(all_files) == EXPECTED_PLOTS_PER_COHORT * 2 * len(
    cohort_outputs
), f"Total plot files mismatch: got {len(all_files)}"
print(
    "\nAlias note: request uses `noise_pred`; implementation column is `noise_preds`."
)
print("feature names parquet:", CACHE_DIR / "feature_names.parquet")
print("figure directory:", FIG_DIR)


[liberman]
 rows: 616
 animals: 105
 fold animal counts: {0: 10, 1: 10, 2: 10, 3: 11, 4: 11, 5: 11, 6: 10, 7: 10, 8: 11, 9: 11}
 interaction pair: ('noise_preds', 'frequency')
 shap parquet: figures/cache/shap/shap_values_liberman.parquet
 metadata parquet: figures/cache/shap/shap_metadata_liberman.parquet

[buran]
 rows: 360
 animals: 57
 fold animal counts: {0: 6, 1: 6, 2: 5, 3: 6, 4: 5, 5: 6, 6: 6, 7: 6, 8: 6, 9: 5}
 interaction pair: ('noise_preds', 'frequency')
 shap parquet: figures/cache/shap/shap_values_buran.parquet
 metadata parquet: figures/cache/shap/shap_metadata_buran.parquet

Alias note: request uses `noise_pred`; implementation column is `noise_preds`.
feature names parquet: figures/cache/shap/feature_names.parquet
figure directory: figures/shap
